In [1]:
import pandas as pd
import numpy as np

# ==========================
# LOAD FILES
# ==========================

files=[

    "cleaned_ADI_Export.xlsx",
    "cleaned_ADIDatabase.xlsx",
    "cleaned_ADIDatabase_2015.xlsx"

]

frames=[]

# ==========================
# STANDARDIZE COLUMN NAMES
# ==========================

def clean_columns(df):

    df.columns=(

        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\n"," ",regex=False)
        .str.replace(r"\s+","_",regex=True)
        .str.lower()

    )

    rename={}

    for col in df.columns:

        c=(
            col
            .replace("_","")
            .replace(" ","")
        )

        # SI NO
        if c in [
            "sino",
            "slno",
            "serialno"
        ]:
            rename[col]="si_no"

        # category
        elif c=="category":
            rename[col]="category"

        # status
        elif c=="status":
            rename[col]="status"

        else:
            rename[col]=col

    df=df.rename(
        columns=rename
    )

    return df


# ==========================
# READ FILES
# ==========================

for file in files:

    print("\nReading:",file)

    df=pd.read_excel(
        file,
        dtype=str
    )

    df=clean_columns(df)

    # remove duplicate cols

    df=df.loc[
        :,
        ~df.columns.duplicated()
    ]

    df["source_file"]=file

    frames.append(df)

# ==========================
# UNION OF COLUMNS
# ==========================

master=[]

for df in frames:

    for col in df.columns:

        if col not in master:

            master.append(col)

# ==========================
# ALIGN FILES
# ==========================

aligned=[]

for df in frames:

    for col in master:

        if col not in df.columns:

            df[col]=np.nan

    aligned.append(
        df[
            master
        ]
    )

# ==========================
# MERGE
# ==========================

merged=pd.concat(
    aligned,
    ignore_index=True
)

print(
    "\nShape Before Duplicate Removal:",
    merged.shape
)

# ==========================
# REMOVE DUPLICATES
# ==========================

dup=merged.duplicated().sum()

merged=merged.drop_duplicates()

print(
    "\nDuplicates Removed:",
    dup
)

# ==========================
# RESET SI_No
# ==========================

remove=[]

for col in merged.columns:

    c=(
        col
        .lower()
        .replace("_","")
    )

    if c in [
        "sino",
        "slno",
        "serialno"
    ]:

        remove.append(col)

merged=merged.drop(
    columns=remove,
    errors="ignore"
)

merged=merged.reset_index(
    drop=True
)

merged.insert(
    0,
    "SI_No",
    np.arange(
        1,
        len(merged)+1
    )
)

# ==========================
# MOVE SOURCE LAST
# ==========================

if "source_file" in merged.columns:

    merged=merged[
        [

            c

            for c in merged.columns

            if c!="source_file"

        ]

        +

        [

            "source_file"

        ]
    ]

# ==========================
# MISSING SUMMARY
# ==========================

missing=(

    merged
    .replace("",np.nan)
    .isna()
    .sum()

)

missing=missing[
    missing>0
]

print("\nMissing Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(merged.shape)

print("\nFinal Columns:")
print(list(merged.columns))

# ==========================
# EXPORT
# ==========================

output="merged_ADI_final.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl"
) as writer:

    merged.to_excel(
        writer,
        index=False,
        sheet_name="Merged"
    )

print("\nSaved:",output)


Reading: cleaned_ADI_Export.xlsx

Reading: cleaned_ADIDatabase.xlsx

Reading: cleaned_ADIDatabase_2015.xlsx

Shape Before Duplicate Removal: (8804, 37)

Duplicates Removed: 0

Missing Summary:
occurrencedate                   757
port                            6221
incidentdetails                   30
incidenteventdetails             705
immediateaction                  113
possibility_recurrence            73
areaofconcern                   1024
detailedextentofdamageinjury    1035
directcause                     1618
indirectcause                   8630
rootcause                       8614
proposedcorrectiveactions        276
corrective_action               2704
leariningpotential              1402
severityofincident               805
revisedincidentcategory         8304
cause_analysis                   252
accident_details                5023
lessionlearnt                   5678
reviewcomments                  6517
closingnote                      977
closingdate                  